In [1]:
from data_frame.analytical.window_function.window_definer import WindowDefiner
from data_frame.spark_utils import get_spark
from pyspark.sql import functions as F
from data_frame.analytical.window_function.rank_calculator import RankCalculator

In [2]:
spark = get_spark(app_name="Window Specifications")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 12:03:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Create sales data
sales_data = [
    ("North", "Electronics", "2024-01-01", 1000),
    ("North", "Electronics", "2024-01-02", 1500),
    ("North", "Clothing", "2024-01-01", 800),
    ("South", "Electronics", "2024-01-01", 2000),
    ("South", "Clothing", "2024-01-02", 1200),
    ("East", "Electronics", "2024-01-01", 1800)
]
df_sales = spark.createDataFrame(sales_data, 
                                 ["region", "category", "date", "revenue"])

## 1. Basic Window Definitions

In [4]:

# Define ordered window
window_spec = WindowDefiner.define_ordered_window(
    partition_cols=["region"],
    order_cols=["revenue"],
    order_direction="desc"
)

In [5]:
# Apply rank function
df_sales = RankCalculator.rank_within_partition(
    df_sales, window_spec, "revenue_rank"
)
print("Rank within region by revenue:")
df_sales.orderBy("region", "revenue_rank").show()

Rank within region by revenue:


+------+-----------+----------+-------+------------+
|region|   category|      date|revenue|revenue_rank|
+------+-----------+----------+-------+------------+
|  East|Electronics|2024-01-01|   1800|           1|
| North|Electronics|2024-01-02|   1500|           1|
| North|Electronics|2024-01-01|   1000|           2|
| North|   Clothing|2024-01-01|    800|           3|
| South|Electronics|2024-01-01|   2000|           1|
| South|   Clothing|2024-01-02|   1200|           2|
+------+-----------+----------+-------+------------+



## 2. Unbounded Window

In [6]:
unbounded_spec = WindowDefiner.define_unbounded_window(
    partition_cols=["region"],
    order_cols=["date"]
)

# Calculate running total
df_sales = df_sales.withColumn(
    "running_total_revenue",
    F.sum("revenue").over(unbounded_spec)
)
print("Running total by region:")
df_sales.orderBy("region", "date").show()

Running total by region:
+------+-----------+----------+-------+------------+---------------------+
|region|   category|      date|revenue|revenue_rank|running_total_revenue|
+------+-----------+----------+-------+------------+---------------------+
|  East|Electronics|2024-01-01|   1800|           1|                 1800|
| North|Electronics|2024-01-01|   1000|           2|                 3300|
| North|   Clothing|2024-01-01|    800|           3|                 3300|
| North|Electronics|2024-01-02|   1500|           1|                 3300|
| South|Electronics|2024-01-01|   2000|           1|                 3200|
| South|   Clothing|2024-01-02|   1200|           2|                 3200|
+------+-----------+----------+-------+------------+---------------------+



## 3. Moving Window

In [7]:
moving_spec = WindowDefiner.define_moving_window(
    partition_cols=["region"],
    order_cols=["date"],
    preceding=1,  # Look back 1 row
    following=0   # Current row only
)

# Calculate moving average
df_sales = df_sales.withColumn(
    "moving_avg_revenue",
    F.avg("revenue").over(moving_spec)
)
print("Moving average (previous 2 rows):")
df_sales.orderBy("region", "date").show()

Moving average (previous 2 rows):
+------+-----------+----------+-------+------------+---------------------+------------------+
|region|   category|      date|revenue|revenue_rank|running_total_revenue|moving_avg_revenue|
+------+-----------+----------+-------+------------+---------------------+------------------+
|  East|Electronics|2024-01-01|   1800|           1|                 1800|            1800.0|
| North|Electronics|2024-01-01|   1000|           2|                 3300|            1000.0|
| North|   Clothing|2024-01-01|    800|           3|                 3300|             900.0|
| North|Electronics|2024-01-02|   1500|           1|                 3300|            1150.0|
| South|Electronics|2024-01-01|   2000|           1|                 3200|            2000.0|
| South|   Clothing|2024-01-02|   1200|           2|                 3200|            1600.0|
+------+-----------+----------+-------+------------+---------------------+------------------+

